# TradeFlow AI — nb5_xgboost

**Fix #6**: Ditingkatkan dari 3 fitur menjadi 32 fitur sesuai PRD §11.
Maritime signals (vessel_voyage_match, port_congestion_index) disertakan sebagai stub — akan diisi dengan data nyata saat sistem produksi berjalan.


In [ ]:
!pip install -q xgboost pandas numpy scikit-learn


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

print('Generating mock dataset: 5,000 historical declarations with 32 features...')
np.random.seed(42)
n = 5000

# === CORE DOCUMENT FEATURES (12) ===
crs_score        = np.clip(np.random.normal(70, 15, n), 0, 100)
correction_count = np.random.poisson(1.2, n)
agent_agreement  = np.random.uniform(0.55, 1.0, n)
hs_confidence    = np.random.uniform(0.4, 1.0, n)   # ChromaDB RAG top-1 score
field_low_count  = np.random.poisson(0.8, n)         # Fields flagged LOW confidence
field_missing_count = np.random.poisson(0.3, n)
has_hs_code      = np.random.binomial(1, 0.85, n)    # 15% docs lack HS code
doc_page_count   = np.random.choice([1, 2, 3], n, p=[0.5, 0.35, 0.15])
has_watermark    = np.random.binomial(1, 0.7, n)
is_scanned       = np.random.binomial(1, 0.4, n)
carrier_code     = np.random.choice([0,1,2,3,4], n)  # HLCU=0, MSCU=1, MAEU=2, EGLV=3, CSLU=4
incoterm_code    = np.random.choice([0,1,2], n)       # CFR=0, FOB=1, CIF=2

# === CEISA VALIDATION FEATURES (8) ===
nib_valid        = np.random.binomial(1, 0.9, n)
npwp_valid       = np.random.binomial(1, 0.88, n)
hs_format_valid  = np.random.binomial(1, 0.92, n)    # 8-digit format
date_parseable   = np.random.binomial(1, 0.95, n)
weight_converted = np.random.binomial(1, 0.93, n)    # MTS/KGS/KGM resolved
container_iso    = np.random.binomial(1, 0.91, n)    # ISO 6346 compliant
cross_doc_rules_pass = np.clip(np.random.normal(8, 1.5, n), 0, 11).astype(int)
insw_flag        = np.random.binomial(1, 0.05, n)    # 5% declarations trigger INSW

# === MARITIME SIGNALS (7) — Stub: populated from AIS/Vessel data in production ===
vessel_voyage_match    = np.random.binomial(1, 0.88, n)  # AIS cross-check result
port_congestion_index  = np.random.uniform(0, 1, n)      # From port lineup data
carrier_rejection_30d  = np.random.uniform(0.05, 0.35, n) # Historical carrier rejection rate
route_risk_score       = np.random.uniform(0, 1, n)      # Origin-destination risk
vessel_flag_risk       = np.random.binomial(1, 0.03, n)  # High-risk flag state
avg_transit_days       = np.random.normal(21, 7, n)
cargo_category_risk    = np.random.choice([0,1,2], n, p=[0.7, 0.25, 0.05])  # 0=normal,1=restricted,2=DG

# === HISTORICAL FEATURES (5) ===
importer_rejection_rate   = np.random.uniform(0, 0.5, n)
importer_declaration_count = np.random.poisson(50, n)
hs_code_rejection_rate    = np.random.uniform(0, 0.4, n)
avg_operator_correction   = np.random.uniform(0.5, 3, n)
days_since_last_rejection = np.clip(np.random.exponential(30, n), 1, 365)

df = pd.DataFrame({
    # Core
    'crs_score':crs_score, 'correction_count':correction_count, 'agent_agreement':agent_agreement,
    'hs_confidence':hs_confidence, 'field_low_count':field_low_count, 'field_missing_count':field_missing_count,
    'has_hs_code':has_hs_code, 'doc_page_count':doc_page_count, 'has_watermark':has_watermark,
    'is_scanned':is_scanned, 'carrier_code':carrier_code, 'incoterm_code':incoterm_code,
    # CEISA
    'nib_valid':nib_valid, 'npwp_valid':npwp_valid, 'hs_format_valid':hs_format_valid,
    'date_parseable':date_parseable, 'weight_converted':weight_converted, 'container_iso':container_iso,
    'cross_doc_rules_pass':cross_doc_rules_pass, 'insw_flag':insw_flag,
    # Maritime
    'vessel_voyage_match':vessel_voyage_match, 'port_congestion_index':port_congestion_index,
    'carrier_rejection_30d':carrier_rejection_30d, 'route_risk_score':route_risk_score,
    'vessel_flag_risk':vessel_flag_risk, 'avg_transit_days':avg_transit_days,
    'cargo_category_risk':cargo_category_risk,
    # Historical
    'importer_rejection_rate':importer_rejection_rate, 'importer_declaration_count':importer_declaration_count,
    'hs_code_rejection_rate':hs_code_rejection_rate, 'avg_operator_correction':avg_operator_correction,
    'days_since_last_rejection':days_since_last_rejection,
})

# Label generation (correlated with risk signals)
risk = (
    (100 - crs_score) * 0.02 +
    correction_count * 0.3 +
    (1 - agent_agreement) * 2 +
    (1 - hs_confidence) * 1.5 +
    insw_flag * 2 +
    (1 - vessel_voyage_match) * 1.2 +
    carrier_rejection_30d * 2
)
prob_reject = 1 / (1 + np.exp(-risk + 4))
labels      = np.random.binomial(1, prob_reject)
df['rejected'] = labels

print(f'Dataset: {len(df)} samples, {df.columns.tolist().count(".") + len(df.columns)} features')
print(f'Rejection rate: {labels.mean():.1%}')

X = df.drop('rejected', axis=1)
y = df['rejected']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=X.columns.tolist())
dtest  = xgb.DMatrix(X_test, label=y_test, feature_names=X.columns.tolist())

params = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'logloss'],
    'max_depth': 6,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'scale_pos_weight': (y==0).sum() / (y==1).sum(),  # Class imbalance correction
}

print('Training XGBoost (32 features)...')
evals = [(dtrain, 'train'), (dtest, 'eval')]
bst   = xgb.train(params, dtrain, num_boost_round=200, evals=evals, early_stopping_rounds=20, verbose_eval=50)

preds     = bst.predict(dtest)
auc       = roc_auc_score(y_test, preds)
acc       = accuracy_score(y_test, (preds > 0.5).astype(int))
print(f'\nTest AUC: {auc:.4f} (target: >= 0.75)')
print(f'Test Accuracy: {acc:.4f}')
if auc >= 0.75:
    print('✅ AUC-ROC LULUS NFR target (>= 0.75)')
else:
    print('❌ AUC-ROC belum mencapai target 0.75')

bst.save_model('rejection_predictor.json')
print('\nModel saved to rejection_predictor.json')

# Feature importance
import matplotlib.pyplot as plt
xgb.plot_importance(bst, max_num_features=15, title='Top 15 Rejection Risk Features')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()
